# Thermal Advisor Agent Walkthrough

**Author: A Taylor**

This notebook demonstrates the agentic architecture: a Bedrock foundation model that calls tools (physics simulator, XGBoost classifier, and a scenario data store) instead of a fine-tuned model.

Pipeline:
1. Build the scenario **data store** from the HuggingFace dataset.
2. Train the **XGBoost classifier**.
3. Wire both into a **ToolDispatcher**.
4. Run the **ThermalAgent** (requires AWS Bedrock credentials).

In [ ]:
import sys
sys.path.insert(0, "..")

from src.datastore import ThermalDataStore
from src.strategy_classifier import StrategyClassifier
from src.tools import ToolDispatcher
from src.agent import ThermalAgent

## 1. Build the scenario data store

The data store indexes the 40K thermal scenarios with TF-IDF and serves them via cosine-similarity retrieval. In production this interface can be backed by Amazon Bedrock Knowledge Bases.

In [ ]:
store = ThermalDataStore.from_huggingface()
store.save("../results/thermal_datastore.pkl")
store.query("Indium Phosphide spectrometer Jovian spectral drift", top_k=3)

## 2. Train the strategy classifier

In [ ]:
from datasets import load_dataset

df = load_dataset("Taylor658/deep-space-optical-chip-thermal-dataset", split="train").to_pandas()
clf = StrategyClassifier()
clf.train(df)
clf.save("../results/strategy_classifier.pkl")

## 3. Wire the tools and run the agent

Requires AWS credentials with Bedrock access in your environment or `.env`.

In [ ]:
dispatcher = ToolDispatcher(classifier=clf, datastore=store)
agent = ThermalAgent(dispatcher=dispatcher)

result = agent.run(
    "Instrument: Spectrometer\n"
    "Material: Indium Phosphide\n"
    "Environment: Jovian System\n"
    "Thermal Effect: Spectral Drift\n"
    "What thermal mitigation strategy should be used and why?"
)
print(result["answer"])
result["tool_calls"]